# flash_reranker

# Flash Reranker in RAG Systems

## What is Reranking?

RAG retrieval happens in **two stages**:

In [ ]:
User Query
    ↓
Stage 1: Initial Retrieval (Fast, ~50-100 docs)
    → Vector similarity search
    ↓
Stage 2: Reranking (Precise, top 5-10)
    → Cross-encoder scoring
    ↓
Send Best Results to LLM

**Why?** Initial retrieval is fast but imprecise. Reranking finds the truly best matches.

---

## What is Flash Reranker?

A **lightweight, ultra-fast reranking model** optimized for production RAG systems.

| Feature | Flash Reranker | BERT Cross-Encoder |

|---------|----------------|-------------------|

| **Speed** | 8ms ⚡ | 450ms |

| **Model Size** | 100MB | 1.2GB |

| **Accuracy** | 95% | 100% |

**Trade-off:** 5% accuracy loss for 50x speed improvement.

---

## How Flash Reranker Works Internally

### Model Architecture

**Base Model:** `ms-marco-MiniLM-L-12-v2` (default)

- **Type**: Distilled cross-encoder

- **Architecture**: MiniLM (lightweight version of BERT)

- **Size**: 120MB (vs 1.2GB for BERT-Large)

- **Layers**: 12 transformer layers (vs 24 in BERT-Large)

### Knowledge Distillation Process

In [ ]:
Teacher Model (BERT-Large)    →    Student Model (MiniLM)
   1.2GB, Slow, 100% accuracy       120MB, Fast, 95% accuracy

**How distillation works:**

1. **Teacher Model** (BERT-Large) scores query-document pairs with high accuracy

2. **Student Model** (MiniLM) learns to mimic teacher's scoring behavior

3. **Result**: 90% compression, 50x speedup, 95% quality retention

### Cross-Encoder Architecture

In [ ]:
**Unlike bi-encoders** (embed query and doc separately):
Cross-Encoder:
[CLS] Query [SEP] Document [SEP] → Transformer → Relevance Score (0-1)
       ↑                ↑
   Jointly encoded (understands interaction)

**Why it's better:**

- Sees query and document together (not separately)

- Captures semantic relationships

- Understands context and nuance

### Optimizations for Speed

**1. Model Compression:**

- **Quantization**: INT8 instead of FP32 (4x memory reduction)

- **Pruning**: Removes unnecessary weights

- **Distillation**: Learns from larger model

**2. Inference Optimizations:**

- **ONNX Runtime**: Optimized execution

- **Batch Processing**: Process multiple docs at once

- **Early Exit**: Stop computation when confident

**3. Architecture Choices:**

- **12 layers** instead of 24 (50% smaller)

- **Smaller hidden size**: 384 vs 768

- **Efficient attention**: Reduces computation

---

## How Reranking Works

In [ ]:
**Problem with vector search:**
Query: "How to fix Python import errors?"

Doc 1: "Python import errors occur when modules not found. Use pip install."
Doc 2: "JavaScript has different error handling than Python programming."

# Both score ~0.75 (both mention "Python" and "errors")
# But Doc 1 is clearly better!

In [ ]:
**Solution - Cross-Encoder:**
Query + Document → [Flash Reranker] → Relevance Score (0-1)

Understands query-document interaction, not just keyword overlap.

### Internal Scoring Process

In [ ]:
**For each document separately:**
Document 1: [CLS] Query [SEP] Document1 [SEP] → Score1
Document 2: [CLS] Query [SEP] Document2 [SEP] → Score2
Document 3: [CLS] Query [SEP] Document3 [SEP] → Score3
...
Document N: [CLS] Query [SEP] DocumentN [SEP] → ScoreN

**Process:**

1. Loop through 50 retrieved documents

2. For each doc: Create [CLS] Query [SEP] Document [SEP]

3. Pass through transformer → Get relevance score (0-1)

4. Sort all 50 scores

5. Return top 5 highest scoring documents

**Batch processing for speed:**

In [ ]:
# What happens internally:
input = "[CLS] Query [SEP] Document [SEP]"
                ↓
    Tokenization (512 max tokens)
                ↓
    12 Transformer Layers
                ↓
    [CLS] token representation
                ↓
    Linear Layer → Sigmoid
                ↓
    Relevance Score: 0.0 - 1.0

---

## Available Flash Reranker Models

| Model | Size | Layers | Speed | Quality | Use Case |

|-------|------|--------|-------|---------|----------|

| **ms-marco-MiniLM-L-12-v2** | 120MB | 12 | Fastest | 95% | General (default) ⭐ |

| **ms-marco-MiniLM-L-6-v2** | 80MB | 6 | Ultra-fast | 90% | Speed-critical |

| **bge-reranker-base** | 278MB | 12 | Fast | 97% | Higher accuracy |

| **ms-marco-MultiBERT-L-12** | 470MB | 12 | Medium | 98% | Multilingual |

In [ ]:
**Model Selection:**
from flashrank import Ranker

# Default (best balance)
ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")

# Ultra-fast (lower accuracy)
ranker = Ranker(model_name="ms-marco-MiniLM-L-6-v2")

# Higher accuracy (slower)
ranker = Ranker(model_name="bge-reranker-base")

### Training Data

**MS MARCO Dataset:**

- 8.8M query-document pairs

- Real Bing search queries

- Human-annotated relevance labels

- Trained to predict: relevant (1) vs not relevant (0)

---

## Flash Reranker vs RRF

**Different purposes:**

| Aspect | Flash Reranker | RRF (Reciprocal Rank Fusion) |

|--------|---------------|------------------------------|

| **Purpose** | Score document relevance | Merge multiple retrievers |

| **Input** | Query + documents | Multiple ranked lists |

| **Method** | ML cross-encoder | Math formula: 1/(k+rank) |

| **Model** | Neural network (120MB) | None (algorithmic) |

| **Speed** | 8ms | Instant |

| **Use Case** | Single retriever refinement | Hybrid search (vector+BM25) |

In [ ]:
**Can use both together:**
RRF (combine vector + BM25) → Flash Reranker (pick best)
       ↓ 50 docs                    ↓ 5 docs

---

## When to Use Flash Reranker?

**Use when:**

- Production chatbot (need <100ms latency)

- Customer support AI (high throughput)

- 95% accuracy is sufficient

**Avoid when:**

- Medical/legal RAG (need maximum accuracy)

- Small query volume (heavy reranker is fine)

---

## Code Example

In [ ]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

# Step 1: Initial retrieval (50 docs)
retriever = vectorstore.as_retriever(search_kwargs={"k": 50})

# Step 2: Rerank with Flash Reranker
compressor = FlashrankRerank(
    model="ms-marco-MiniLM-L-12-v2"  # Specify model
)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

# Step 3: Get top 5 best matches
docs = compression_retriever.invoke("What is Python?")

---

## Key Takeaways

1. **Two-stage retrieval**: Fast initial search → Precise reranking

2. **Distilled Model**: MiniLM learns from BERT-Large (95% quality, 10x smaller)

3. **Cross-Encoder**: Jointly encodes query+doc for better understanding

4. **Optimizations**: INT8 quantization, pruning, ONNX runtime

5. **Default Model**: ms-marco-MiniLM-L-12-v2 (120MB, 12 layers)

6. **vs RRF**: Reranker scores relevance, RRF merges retrievers

7. **Impact**: 40% improvement in answer relevance

---

In [ ]:
# Verify flashrank installation
try:
    from flashrank import Ranker
    print("✅ flashrank is installed and working!")
    print("You can now use FlashrankRerank in your code.")
except ImportError as e:
    print(f"❌ flashrank NOT found: {e}")
    print("\nPlease follow the installation steps above in a NEW terminal.")
    print("After installation, RESTART this Jupyter kernel.")

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
from langchain_openai import ChatOpenAI

api_key = os.environ['UNIFIED_LLM_KEY']
# print(api_key)
base_url = ""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=256,
    api_key=api_key,
    base_url=base_url
)

In [ ]:
# Helper function for printing docs


def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [
                f"Document {i + 1}:\n\n{d.page_content}\nMetadata: {d.metadata}"
                for i, d in enumerate(docs)
            ]
        )
    )

In [ ]:
from langchain.embeddings.base import Embeddings
from pydantic import BaseModel
from typing import Any, List, Mapping, Optional
import requests
import numpy as np  # Added missing import


class MLServerEmbedding(Embeddings, BaseModel):
    max_batch_size: int = 32
    url: str

    """ public """

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        text_num = len(texts)
        result = []
        for i in range(0, text_num, self.max_batch_size):
            result += self._get_embedding(texts[i: i + self.max_batch_size])
        return result

    def embed_query(self, text: str) -> List[float]:
        return self._get_embedding([text])[0]

    """ private """

    @staticmethod
    def _wrap_payload(text_list):
        return {
            "inputs": [
                {
                    "name": "input",
                    "shape": [len(text_list)],
                    "datatype": "str",
                    "data": text_list
                }
            ]
        }

    @staticmethod
    def _parse_response(response):
        if response.status_code != 200:
            raise Exception(response)
        outputs = response.json()["outputs"][0]
        return np.array(outputs["data"]).reshape(outputs["shape"]).tolist()

    def _get_embedding(self, text_list: List[str]) -> List[List[float]]:
        return self._parse_response(requests.post(url=self.url,
                                                  json=self._wrap_payload(text_list),
                                                  headers={"Content-Type": "application/json"},
                                                  params={}))

In [ ]:
embedding_model = "all-mpnet-base--fccb5"
embedding_url = ""+ embedding_model +"/v2/models/"+ embedding_model + "/infer"
embeddings = MLServerEmbedding(url=embedding_url)

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = TextLoader(
    "/Users/vinotganesan/Learning/LLM & AGENTS/RAG/files/state_of_the_union.txt",
).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)
for idx, text in enumerate(texts):
    text.metadata["id"] = idx

# embedding = OpenAIEmbeddings(model="text-embedding-ada-002")
retriever = FAISS.from_documents(texts, embeddings).as_retriever(search_kwargs={"k": 20})

query = "What did the president say about Ketanji Brown Jackson"
docs = retriever.invoke(query)
pretty_print_docs(docs)

# Performing reranking with FlashRank

In [ ]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank
from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(temperature=0)

from flashrank import Ranker 

ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")

compressor = FlashrankRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "What did the president say about Ketanji Jackson Brown"
)
print([doc.metadata["id"] for doc in compressed_docs])

**After reranking, the top 3 documents are different from the top 3 documents retrieved by the base retriever.**

In [ ]:
pretty_print_docs(compressed_docs)

In [ ]:
from langchain.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)

chain.invoke(query)